In [ ]:
import pandas as pd
import os
import networkx as nx
import matplotlib.pyplot as plt
import math
import torch
from torch_geometric.data import Data
import itertools
import re
from collections import Counter
import gensim
import numpy as np
import scipy.sparse as sp
import pickle
import csv
import datetime
import json

In [ ]:
torch.cuda.is_available()

In [ ]:
BASE_PATH = "Fakeddit/"

### Loading the source posts

In [ ]:
df=pd.read_csv(os.path.join(BASE_PATH,"all_train.tsv"), sep='\t', header=0)
df2=pd.read_csv(os.path.join(BASE_PATH,"all_validate.tsv"), sep='\t', header=0)
df3=pd.read_csv(os.path.join(BASE_PATH,"all_test_public.tsv"), sep='\t', header=0)

In [ ]:
df.head()

In [ ]:
## Combining train,test and valid into one dataframe and clearing those don't have any text in it
combined_df=pd.DataFrame()
combined_df = pd.concat([df,df2,df3], ignore_index=True)
print(len(combined_df))
combined_df = combined_df.dropna(subset=["clean_title"])
print(len(combined_df))
combined_df = combined_df.reset_index(drop=True)

### Loading the Comments

In [ ]:
comments=pd.read_csv(os.path.join(BASE_PATH,"all_comments.tsv"), sep='\t', header=0)

In [ ]:
comments.head()

### Mapping Source posts to comment chains

In [ ]:
top_post_ids=[i[3:] for i in comments["parent_id"] if str(i)!='nan' and i[0:3]=='t3_']
top_post_ids = list(set(top_post_ids))

In [ ]:
print(len(top_post_ids))

In [ ]:
mask = combined_df["id"].isin(top_post_ids)
all_top_posts = combined_df[mask]
print(len(all_top_posts))

In [ ]:
all_top_posts=all_top_posts[['id','created_utc','author','clean_title','2_way_label','num_comments']]

In [ ]:
times_mapping = dict()
for i,j in zip(all_top_posts["id"],all_top_posts["created_utc"]):
    dt_object = datetime.datetime.fromtimestamp(j)
    times_mapping[i]=dt_object.strftime("%Y-%m-%d %H:%M:%S")

In [ ]:
id_times_sorted = {k: v for k, v in sorted(times_mapping.items(), key=lambda item: item[1])} 

In [ ]:
print(len(id_times_sorted))

In [ ]:
## Splitting the dataset
list_ids = [i for i in id_times_sorted]
n_1 = int(0.7 * len(list_ids))
n = int(0.8 * len(list_ids))
print(n)

train_ids = list_ids[:n_1]
valid_ids = list_ids[n_1:n]
test_ids = list_ids[n:]

In [ ]:
print(len(train_ids),len(valid_ids),len(test_ids))

In [ ]:
### Use this split for further steps
with open("train_ids.pkl","wb") as f:
     pickle.dump(train_ids,f)

with open("valid_ids.pkl","wb") as f:
     pickle.dump(valid_ids,f)

with open("test_ids.pkl","wb") as f:
     pickle.dump(test_ids,f)

In [ ]:
# Considering only the train split for graph construction
mask = all_top_posts["id"].isin(train_ids)
all_top_posts = all_top_posts[mask]

In [ ]:
uni=list(all_top_posts["id"].unique())
print(len(uni))
comm=comments[comments["submission_id"].isin(uni)]
comm.loc[:,'parent_id'] = comm['parent_id'].copy().str[3:]
grp=comm.groupby(["submission_id"])

### Creating the Graph

In [ ]:
## Map Id to User
id_to_user = dict()
for id_,user in zip(all_top_posts["id"],all_top_posts["author"]):
    id_to_user[id_]=user

for name,group in grp:
    for id_,user,sent in zip(group["id"],group["author"],group['body']):
        if str(sent)!="nan":
           id_to_user[id_]=user

In [ ]:
train_users=list(set(id_to_user.values()))
print(len(train_users))

In [ ]:
with open("train_user_list.pkl","wb") as f:
    pickle.dump(train_users,f)

In [ ]:
G = nx.Graph()
c=0
for name,group in grp:
    for id_,p_id,sent in zip(group["id"],group["parent_id"],group['body']):
        if str(sent)!="nan" :
           try:
              source = id_to_user[id_]
              target = id_to_user[p_id]
              if source != target and not G.has_edge(source, target) and not G.has_edge(target,source) and str(source)!="nan" and str(target)!="nan":
                 G.add_edge(source, target)
           except Exception as e:
               print(e)
               c+=1

In [ ]:
print("Noises in Dataset : ", c) 

In [ ]:
user_to_id = {}
t = 0

for edge in G.edges():
    for node in edge:
        if node not in user_to_id:
            user_to_id[node] = t
            t += 1


In [ ]:
edge_index = []
for edge in G.edges():
    edge_index.append([user_to_id[edge[0]],user_to_id[edge[1]]])

In [ ]:
edges_final = torch.tensor(edge_index).t().contiguous()

In [ ]:
## Declare the Dimension here to change if needed
from torch_geometric.nn import Node2Vec
import os.path as osp
import torch
from tqdm.notebook import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'  # check if cuda is available to send the model and tensors to the GPU
model = Node2Vec(edges_final, embedding_dim=128, walk_length=20,
                 context_size=10, walks_per_node=10,
                 num_negative_samples=1, p=1, q=1, sparse=True).to(device)

loader = model.loader(batch_size=128, shuffle=True, num_workers=4)  # data loader to speed the train 
optimizer = torch.optim.SparseAdam(list(model.parameters()), lr=0.01)  # initzialize the optimizer 

def train():
    model.train()  # put model in train model
    total_loss = 0
    for pos_rw, neg_rw in tqdm(loader):
        optimizer.zero_grad()  # set the gradients to 0
        loss = model.loss(pos_rw.to(device), neg_rw.to(device))  # compute the loss for the batch
        loss.backward()
        optimizer.step()  # optimize the parameters
        total_loss += loss.item()
    return total_loss / len(loader)


for epoch in range(1, 51):
    loss = train()
    print(f'Epoch: {epoch:02d}, Loss: {loss:.4f}')



In [ ]:
all_vectors = []
for tensor in model(torch.arange(G.number_of_nodes(), device=device)):
    all_vectors.append(tensor.detach().cpu().numpy())
# # save the vectors
# with open("vectors.txt", "w") as f:
#     f.write(all_vectors)
# # # save the labels
# # with open("labels.txt", "w") as f:
# #     f.write("\n".join([str(label) for label in data.y.numpy()]))

In [ ]:
with open("user_to_id.json","w") as f:
     json.dump(user_to_id,f)

In [ ]:
with open("User_features.pkl","wb") as f:
      pickle.dump(all_vectors,f)